In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "07-application-agent-framework/retrieval-rag/embeddings-lab/solutions")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# Exercises 04 · ANN internals
Implement the two core moves of IVF-PQ, then demonstrate the filtered-search
trap yourself. Solutions: `solutions/ex04_solutions.ipynb`.

In [1]:
import numpy as np
rng = np.random.default_rng(0)

def l2sq(A, B):
    return (A * A).sum(1)[:, None] + (B * B).sum(1)[None, :] - 2 * A @ B.T

def kmeans(X, k, iters=12, seed=0):
    r = np.random.default_rng(seed)
    C = X[r.choice(len(X), k, replace=False)].copy()
    for _ in range(iters):
        a = np.argmin(l2sq(X, C), axis=1)
        for c in range(k):
            if (a == c).any():
                C[c] = X[a == c].mean(0)
    return C, np.argmin(l2sq(X, C), axis=1)

N, D = 4000, 32
X = rng.normal(size=(N, D)) + rng.normal(size=(20, D))[rng.integers(0, 20, N)] * 3
Q = X[rng.integers(0, N, 50)] + 0.3 * rng.normal(size=(50, D))
gold = np.argsort(l2sq(Q, X), axis=1)[:, :10]

## Task 1 — asymmetric distance computation (ADC)
Given per-subspace `codebooks` (M, 256, sub) and `codes` (N, M), return the
PQ distance from query `q` to every database item. It must equal the exact
squared distance to each item's *reconstruction* — that's the check.

In [2]:
M = 4
sub = D // M
codebooks = np.zeros((M, 256, sub))
codes = np.zeros((N, M), dtype=np.uint8)
for m in range(M):
    codebooks[m], codes[:, m] = kmeans(X[:, m * sub:(m + 1) * sub], 256, seed=m)

def adc_dists(q, codebooks, codes):
    M, _, sub = codebooks.shape
    lut = np.stack([((codebooks[m] - q[m * sub:(m + 1) * sub]) ** 2).sum(1)
                    for m in range(M)])
    return lut[np.arange(M)[None, :], codes].sum(1)

q = Q[0]
recon = np.concatenate([codebooks[m][codes[:, m]] for m in range(M)], axis=1)
exact_to_recon = ((recon - q) ** 2).sum(1)
assert np.allclose(adc_dists(q, codebooks, codes), exact_to_recon, atol=1e-8)
print("adc_dists ✓ (equals distance to reconstructions)")

adc_dists ✓ (equals distance to reconstructions)


## Task 2 — IVF search
`ivf_search(q, n_probe)`: find the `n_probe` nearest coarse centroids, gather
their inverted lists, score exactly, return top-10 ids. With
`n_probe = n_list` it must match brute force.

In [3]:
NLIST = 32
cents, cell = kmeans(X, NLIST, seed=99)
lists = [np.where(cell == c)[0] for c in range(NLIST)]

def ivf_search(q, n_probe):
    probe = np.argsort(((cents - q) ** 2).sum(1))[:n_probe]
    cand = np.concatenate([lists[c] for c in probe])
    d = ((X[cand] - q) ** 2).sum(1)
    return cand[np.argsort(d)[:10]]

assert np.array_equal(np.sort(ivf_search(Q[0], NLIST)), np.sort(gold[0]))
rec4 = np.mean([len(set(ivf_search(q, 4)) & set(g)) / 10 for q, g in zip(Q, gold)])
print(f"ivf_search ✓   recall@10 with n_probe=4: {rec4:.2f}")

ivf_search ✓   recall@10 with n_probe=4: 1.00


## Task 3 — post-filtering collapses on selective filters
Give every vector a random tag (1% selectivity). Implement `post_filter`:
search top-`k_search` *ignoring* tags, then keep matches. Measure recall
against exact filtered search and watch it collapse.

In [4]:
tag = rng.integers(0, 100, N)
want = tag[np.argmin(l2sq(Q, X), axis=1)]

def filtered_gold(q, t):
    idx = np.where(tag == t)[0]
    return idx[np.argsort(((X[idx] - q) ** 2).sum(1))[:10]]

def post_filter(q, t, k_search=100):
    top = np.argsort(((X - q) ** 2).sum(1))[:k_search]
    return top[tag[top] == t][:10]

recs = [len(set(post_filter(q, t)) & set(filtered_gold(q, t))) / 10
        for q, t in zip(Q, want)]
post_recall = float(np.mean(recs))
print(f"post-filter recall@10 = {post_recall:.2f}   (pre-filter = 1.00 by construction)")
assert post_recall < 0.9, "post-filtering should visibly lose recall at 1% selectivity"
print("filtered-search trap ✓ demonstrated")

post-filter recall@10 = 0.20   (pre-filter = 1.00 by construction)
filtered-search trap ✓ demonstrated


## Task 4 (open) — the knee of the curve
For your `ivf_search`, sweep `n_probe ∈ {1..32}` and find the smallest value
reaching ≥ 0.95 recall@10. How does it change if you double `NLIST`? (Rule of
thumb: n_list ≈ √N, then tune n_probe on *your* recall target.)